In [ ]:
#| default_exp compute

# Compute

> EC2 instances, EKS clusters, and ECR container registries.

## Amazon EC2

Equivalent to Azure VM. IMDSv2 enforced, instance profile support.

```python
inst = create_instance(auth, 'my-vm', instance_type='t3.medium')
print(instance_ip(auth, inst['InstanceId']))
```

In [ ]:
#| export
_tags = lambda d: [{'Key': k, 'Value': v} for k, v in (d or {}).items()]

def _ec2(auth):
    return auth.session.client('ec2')

_UBUNTU_FILTER = [
    {'Name': 'name',          'Values': ['ubuntu/images/hvm-ssd/ubuntu-jammy-22.04-amd64-server-*']},
    {'Name': 'architecture',  'Values': ['x86_64']},
    {'Name': 'state',         'Values': ['available']},
    {'Name': 'virtualization-type', 'Values': ['hvm']},
]

def _latest_ubuntu_ami(auth) -> str:
    from fastcore.basics import first
    images = _ec2(auth).describe_images(
        Filters=_UBUNTU_FILTER, Owners=['099720109477'])['Images']
    return first(sorted(images, key=lambda x: x['CreationDate'], reverse=True))['ImageId']

def create_instance(auth, name, instance_type='t3.medium', ami=None, key_name=None,
                    subnet_id=None, sg_ids=None, iam_instance_profile=None,
                    termination_protection=False, tags=None, **compliance_opts) -> dict:
    'Launch EC2 instance with IMDSv2 enforced. termination_protection=True prevents accidental deletion.'
    ec2 = _ec2(auth)
    image_id = ami or _latest_ubuntu_ami(auth)
    tag_specs = [{'ResourceType': 'instance',
                  'Tags': [{'Key': 'Name', 'Value': name}] + _tags(tags)}]
    kwargs = dict(
        ImageId=image_id,
        InstanceType=instance_type,
        MinCount=1, MaxCount=1,
        MetadataOptions={'HttpTokens': 'required',  # IMDSv2
                         'HttpPutResponseHopLimit': 1},
        DisableApiTermination=termination_protection,
        TagSpecifications=tag_specs,
    )
    if key_name:  kwargs['KeyName'] = key_name
    if subnet_id: kwargs['SubnetId'] = subnet_id
    if sg_ids:    kwargs['SecurityGroupIds'] = sg_ids
    if iam_instance_profile:
        kwargs['IamInstanceProfile'] = {'Name': iam_instance_profile}
    return ec2.run_instances(**kwargs)['Instances'][0]

def instance_ip(auth, instance_id) -> str:
    'Return the public IP address of an EC2 instance (falls back to private IP).'
    from fastcore.basics import first
    inst = first(_ec2(auth).describe_instances(
        InstanceIds=[instance_id])['Reservations'])['Instances'][0]
    return inst.get('PublicIpAddress', inst.get('PrivateIpAddress', ''))

def start_instance(auth, instance_id):
    'Start a stopped EC2 instance.'
    _ec2(auth).start_instances(InstanceIds=[instance_id])

def stop_instance(auth, instance_id):
    'Stop a running EC2 instance.'
    _ec2(auth).stop_instances(InstanceIds=[instance_id])

def terminate_instance(auth, instance_id):
    'Terminate (permanently delete) an EC2 instance.'
    _ec2(auth).terminate_instances(InstanceIds=[instance_id])


## Amazon EKS

Equivalent to Azure AKS. Managed Kubernetes with node group.

```python
create_eks(auth, 'my-cluster', node_count=3)
print(eks_kubeconfig(auth, 'my-cluster'))
```

In [ ]:
#| export
import base64, yaml

def _eks(auth):
    return auth.session.client('eks')

def create_eks(auth, name, node_type='m5.large', node_count=3,
               version='1.31', subnet_ids=None, sg_ids=None,
               logging_types=None, endpoint_public_access=True,
               endpoint_private_access=False, tags=None, **compliance_opts) -> dict:
    'Create EKS cluster with managed node group and control-plane logging.'
    from .network import create_role, attach_policy
    client = _eks(auth)

    cluster_role = create_role(
        auth, f'{name}-eks-cluster-role',
        trust_policy=_eks_cluster_trust())
    attach_policy(auth, f'{name}-eks-cluster-role',
                  'arn:aws:iam::aws:policy/AmazonEKSClusterPolicy')

    node_role = create_role(
        auth, f'{name}-eks-node-role',
        trust_policy=_ec2_trust())
    for policy in ['AmazonEKSWorkerNodePolicy',
                   'AmazonEKS_CNI_Policy',
                   'AmazonEC2ContainerRegistryReadOnly']:
        attach_policy(auth, f'{name}-eks-node-role',
                      f'arn:aws:iam::aws:policy/{policy}')

    resources_vpc = {'endpointPublicAccess': endpoint_public_access,
                     'endpointPrivateAccess': endpoint_private_access}
    if subnet_ids: resources_vpc['subnetIds'] = subnet_ids
    if sg_ids:     resources_vpc['securityGroupIds'] = sg_ids

    log_types = logging_types or ['api', 'audit', 'authenticator']
    tag_dict = {t['Key']: t['Value'] for t in _tags(tags)} if tags else {}
    try:
        cluster = client.create_cluster(
            name=name,
            version=version,
            roleArn=cluster_role['Role']['Arn'],
            resourcesVpcConfig=resources_vpc,
            logging={'clusterLogging': [{'types': log_types, 'enabled': True}]},
            tags=tag_dict,
        )['cluster']
    except client.exceptions.ResourceInUseException:
        cluster = client.describe_cluster(name=name)['cluster']

    if subnet_ids:
        try:
            client.create_nodegroup(
                clusterName=name,
                nodegroupName=f'{name}-ng',
                scalingConfig={'minSize': 1, 'maxSize': node_count * 2,
                               'desiredSize': node_count},
                instanceTypes=[node_type],
                nodeRole=node_role['Role']['Arn'],
                subnets=subnet_ids,
            )
        except client.exceptions.ResourceInUseException:
            pass

    return cluster

def eks_kubeconfig(auth, name) -> str:
    'Build and return kubeconfig YAML for an EKS cluster using boto3 (no CLI required).'
    cluster = _eks(auth).describe_cluster(name=name)['cluster']
    endpoint = cluster['endpoint']
    ca_data = cluster['certificateAuthority']['data']
    cluster_arn = cluster['arn']
    kubeconfig = {
        'apiVersion': 'v1',
        'kind': 'Config',
        'clusters': [{'name': cluster_arn,
                      'cluster': {'server': endpoint,
                                  'certificate-authority-data': ca_data}}],
        'contexts': [{'name': cluster_arn,
                      'context': {'cluster': cluster_arn, 'user': cluster_arn}}],
        'current-context': cluster_arn,
        'users': [{'name': cluster_arn,
                   'user': {'exec': {
                       'apiVersion': 'client.authentication.k8s.io/v1beta1',
                       'command': 'aws',
                       'args': ['eks', 'get-token', '--cluster-name', name,
                                '--region', auth.region],
                   }}}],
    }
    return yaml.dump(kubeconfig, default_flow_style=False)

def scale_eks(auth, name, node_count):
    'Update desired node count on the default node group.'
    _eks(auth).update_nodegroup_config(
        clusterName=name,
        nodegroupName=f'{name}-ng',
        scalingConfig={'desiredSize': node_count})

def _eks_cluster_trust() -> dict:
    return {'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'eks.amazonaws.com'},
        'Action': 'sts:AssumeRole'}]}

def _ec2_trust() -> dict:
    return {'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'ec2.amazonaws.com'},
        'Action': 'sts:AssumeRole'}]}


## Amazon ECR

Equivalent to Azure Container Registry. Scan-on-push enabled by default.

```python
create_ecr(auth, 'my-app')
print(ecr_login_url(auth, 'my-app'))
```

In [ ]:
#| export
def _ecr(auth):
    return auth.session.client('ecr')

def create_ecr(auth, name, scan_on_push=True, lifecycle_policy=True, tags=None) -> dict:
    'Create ECR repository with scan-on-push. lifecycle_policy=True removes untagged images after 30 days.'
    import json as _json
    client = _ecr(auth)
    try:
        repo = client.create_repository(
            repositoryName=name,
            imageScanningConfiguration={'scanOnPush': scan_on_push},
            encryptionConfiguration={'encryptionType': 'AES256'},
            tags=_tags(tags),
        )['repository']
    except client.exceptions.RepositoryAlreadyExistsException:
        repo = client.describe_repositories(repositoryNames=[name])['repositories'][0]
    if lifecycle_policy:
        client.put_lifecycle_policy(
            repositoryName=name,
            lifecyclePolicyText=_json.dumps({
                'rules': [{
                    'rulePriority': 1,
                    'description': 'Remove untagged images after 30 days',
                    'selection': {'tagStatus': 'untagged',
                                  'countType': 'sinceImagePushed',
                                  'countUnit': 'days', 'countNumber': 30},
                    'action': {'type': 'expire'},
                }],
            }),
        )
    return repo

def ecr_login_url(auth, name) -> str:
    'Return the ECR repository URI (host/name for docker push/pull).'
    return f'{auth.account_id}.dkr.ecr.{auth.region}.amazonaws.com/{name}'

def attach_ecr_to_eks(auth, ecr_name, eks_name):
    'Grant EKS node role AmazonEC2ContainerRegistryReadOnly access to ECR.'
    from .network import attach_policy
    attach_policy(auth, f'{eks_name}-eks-node-role',
                  'arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly')
